LSTM CODE

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import os
from tabulate import tabulate
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
import warnings
import datetime as dt

warnings.filterwarnings("ignore")

# Function to create dataset for time-series prediction
def create_dataset(dataset, time_step=1):
    dataX, dataY = [], []
    for i in range(len(dataset) - time_step - 1):
        a = dataset[i:(i + time_step), 0]
        dataX.append(a)
        dataY.append(dataset[i + time_step, 0])
    return np.array(dataX), np.array(dataY)

# Function to fetch company name from Yahoo Finance
def get_company_name(index_name):
    """
    Fetches the company name from Yahoo Finance using the index name as the ticker symbol.

    Args:
        index_name (str): The index name (ticker symbol) to search for.

    Returns:
        str: The company name if found, otherwise returns the original index name.
    """
    try:
        ticker = yf.Ticker(index_name)
        company_name = ticker.info.get('longName', index_name)  # Use longName if available, else default to index_name
        return company_name
    except Exception as e:
        print(f"Error fetching company name for {index_name}: {e}")
        return index_name

# Main function to get predicted values
def getpredictedvalues(selectedscript_1, start_date='2021-01-01', end_date='2025-01-01'):
    """
    Main function to calculate predicted values using LSTM model.

    Args:
        selectedscript_1 (pd.DataFrame): DataFrame containing historical stock data with a 'Date' column and at least a 'Close' column.
        start_date (str, optional): Start date for filtering data ('YYYY-MM-DD'). Defaults to '2021-01-01'.
        end_date (str, optional): End date for filtering data ('YYYY-MM-DD'). Defaults to '2025-01-01'.

    Returns:
        tuple: A tuple containing:
            - df (pd.DataFrame): DataFrame with model information (Model, Train R2 Score, Test R2 Score).
            - finaldf (pd.DataFrame): DataFrame with predicted 'Close' values.
            - selectedscript (pd.DataFrame): The original processed DataFrame used for training.
    """
    # Data Cleaning and Preparation
    selectedscript_2 = selectedscript_1.dropna().reset_index(drop=True)
    selectedscript = selectedscript_2.copy()
    selectedscript['Date'] = pd.to_datetime(selectedscript['Date'], format='%Y-%m-%d')

    # Date Range Filtering
    if start_date and end_date:
        selectedscript = selectedscript[(selectedscript['Date'] >= start_date) & (selectedscript['Date'] <= end_date)]

    selectedscript = selectedscript.set_index('Date')
    close_df = selectedscript[['Close']].reset_index()
    close_stock = close_df.copy()
    del close_df['Date']
    # Scaling the Data
    scaler = MinMaxScaler(feature_range=(0, 1))
    closedf = scaler.fit_transform(np.array(close_df).reshape(-1, 1))

    # Splitting Data into Training and Testing Sets
    training_size = int(len(closedf) * 0.80)
    test_size = len(closedf) - training_size
    train_data, test_data = closedf[0:training_size, :], closedf[training_size:len(closedf), :1]

    # Creating Dataset with Time Steps
    time_step = 13
    X_train, y_train = create_dataset(train_data, time_step)
    X_test, y_test = create_dataset(test_data, time_step)

    # Reshape Input for LSTM
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # Build LSTM Model
    model = Sequential([
        LSTM(32, return_sequences=True, input_shape=(time_step, 1)),
        LSTM(32, return_sequences=True),
        LSTM(32),
        Dense(1)
    ])
    model.compile(loss='mean_squared_error', optimizer='adam')

    # Train the Model
    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=32, verbose=1)

    # Make Predictions
    train_predict = model.predict(X_train)
    test_predict = model.predict(X_test)

    # Inverse Transform Predictions
    train_predict = scaler.inverse_transform(train_predict)
    test_predict = scaler.inverse_transform(test_predict)
    original_ytrain = scaler.inverse_transform(y_train.reshape(-1, 1))
    original_ytest = scaler.inverse_transform(y_test.reshape(-1, 1))

    # Calculate R2 Score
    train_r2_lstm = r2_score(original_ytrain, train_predict)
    test_r2_lstm = r2_score(original_ytest, test_predict)

    # Predict Future Values
    x_input = test_data[len(test_data) - time_step:].reshape(1, -1)
    temp_input = list(x_input)[0].tolist()
    lst_output = []
    n_steps = time_step
    pred_days = 5
    i = 0
    while i < pred_days:
        if len(temp_input) > time_step:
            x_input = np.array(temp_input[1:]).reshape(1, -1)
            x_input = x_input.reshape((1, n_steps, 1))
            yhat = model.predict(x_input, verbose=0)
            temp_input.extend(yhat[0].tolist())
            temp_input = temp_input[1:]
            lst_output.extend(yhat.tolist())
            i += 1
        else:
            x_input = x_input.reshape((1, n_steps, 1))
            yhat = model.predict(x_input, verbose=0)
            temp_input.extend(yhat[0].tolist())
            lst_output.extend(yhat.tolist())
            i += 1

    # Create Final DataFrames
    lstmdf = scaler.inverse_transform(np.array(closedf.tolist() + lst_output).reshape(-1, 1)).flatten().tolist()
    finaldf = pd.DataFrame({'Close': lstmdf})
    data = {"Model": ["LSTM"], "Train R2 Score": [train_r2_lstm], "Test R2 Score": [test_r2_lstm]}
    df = pd.DataFrame(data)

     # Calculate the predicted High and Low values for the next 5 days
    last_date = selectedscript.index[-1]
    future_dates = [last_date + pd.DateOffset(days=i) for i in range(1, 6)]
    future_preds = finaldf['Close'].tail(5).values
    
    # Since actual High and Low data is not predicted by the LSTM model in this code,
    # we will use a simple heuristic: assume High is 5% above and Low is 5% below the predicted Close.
    future_high = future_preds * 1.05
    future_low = future_preds * 0.95
    
    return df, finaldf, selectedscript, train_r2_lstm, test_r2_lstm, future_preds, future_high, future_low

# Main script execution
if __name__ == "__main__":
    # Paths and configurations
    file_path = 'C://Users//manoj//Downloads//Major project data//Major pro source codes//DATASETS//filtered_indices_output.csv'
    daily_data_path = 'C://Users//manoj//Downloads//Major project data//Major pro source codes//DATASETS//Daily_data'
    output_csv_file = 'C://Users//manoj//Downloads//Major project data//Major pro source codes//DATASETS//lstm_prediction_output.csv'
    sectors_file_path = 'C://Users//manoj//Downloads//Major project data//Major pro source codes//DATASETS//indicesstocks.csv'

    try:
        # Load selected indices
        if os.path.exists(file_path):
            print(f"File found: {file_path}")
            selected_indices = pd.read_csv(file_path, on_bad_lines='skip')  # Handle bad lines
        else:
            print(f"File not found: {file_path}")
            raise FileNotFoundError(f"The specified file does not exist: {file_path}")

        # Load sectors file
        if os.path.exists(sectors_file_path):
            sectors_df = pd.read_csv(sectors_file_path, on_bad_lines='skip')  # Handle bad lines
        else:
            print(f"File not found: {sectors_file_path}")
            sectors_df = pd.DataFrame()  # Create an empty DataFrame to avoid errors

        all_output_data = []
        unique_index_codes = selected_indices['indexcode'].unique()
        current_date = dt.datetime.now().strftime("%Y-%m-%d")

        for index_code in unique_index_codes:
            filtered_indices = selected_indices[selected_indices['indexcode'] == index_code]
            for _, row in filtered_indices.iterrows():
                index_name = row['indexname']

                # Fetch company name from Yahoo Finance
                company_name = get_company_name(index_name)
                daily_file_name = f"{index_name.replace('.', '_')}.csv"
                daily_file_path = os.path.join(daily_data_path, daily_file_name)

                try:
                    daily_data = pd.read_csv(daily_file_path)

                    # Call the getpredictedvalues function and unpack all returned values
                    df, finaldf, selectedscript, train_r2, test_r2, future_preds, future_high, future_low = getpredictedvalues(daily_data)

                    output_data = {
                        'Run Date': current_date,
                        'Index Name': index_name,
                        'Company Name': company_name,
                        'Model': 'LSTM',
                        'Train R2 Score': train_r2,
                        'Test R2 Score': test_r2,
                        'Close Day 1': future_preds[0],
                        'Close Day 2': future_preds[1],
                        'Close Day 3': future_preds[2],
                        'Close Day 4': future_preds[3],
                        'Close Day 5': future_preds[4],
                        'High Day 1': future_high[0],
                        'High Day 2': future_high[1],
                        'High Day 3': future_high[2],
                        'High Day 4': future_high[3],
                        'High Day 5': future_high[4],
                        'Low Day 1': future_low[0],
                        'Low Day 2': future_low[1],
                        'Low Day 3': future_low[2],
                        'Low Day 4': future_low[3],
                        'Low Day 5': future_low[4]
                    }
                    # Check for duplicates BEFORE appending
                    duplicate = False
                    for existing_data in all_output_data:
                        if existing_data['Run Date'] == output_data['Run Date'] and existing_data['Index Name'] == output_data['Index Name']:
                            duplicate = True
                            break
                    if not duplicate:
                        all_output_data.append(output_data)
                    else:
                        print(f"Duplicate entry found for {index_name} on {current_date}. Skipping.")

                except Exception as e:
                    print(f"Error processing {index_name}: {str(e)}")

        # Define columns to maintain order
        columns = ['Run Date', 'Index Name', 'Company Name', 'Model', 'Train R2 Score', 'Test R2 Score',
                   'Close Day 1', 'Close Day 2', 'Close Day 3', 'Close Day 4', 'Close Day 5',
                   'High Day 1', 'High Day 2', 'High Day 3', 'High Day 4', 'High Day 5',
                   'Low Day 1', 'Low Day 2', 'Low Day 3', 'Low Day 4', 'Low Day 5']
        output_df = pd.DataFrame(all_output_data, columns=columns)
        output_df.to_csv(output_csv_file, index=False)

        print(f"//nPredictions saved to {output_csv_file}")
        print("//nFinal Output:")
        print(tabulate(output_df, headers='keys', tablefmt='fancy_grid', showindex=False))

    except Exception as e:
        print(f"An error occurred: {str(e)}")


File found: C://Users//manoj//Downloads//Major project data//Major pro source codes//DATASETS//filtered_indices_output.csv
Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 72ms/step - loss: 0.0148 - val_loss: 0.0038
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 6.1356e-04 - val_loss: 0.0035
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 3.7648e-04 - val_loss: 0.0123
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 3.5216e-04 - val_loss: 0.0050
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 2.9512e-04 - val_loss: 0.0035
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 3.0861e-04 - val_loss: 0.0053
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 2.6011e-04 - val_loss: 0.0035
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 2.7393e-04 - val_loss: 0.0030
Epoch 9/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - loss: 3.2037e-04 - val_loss: 0.0030
Epoch 10/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 3.3375e-04 - val_los